In [1]:
import pandas as pd

df = pd.read_csv("../data/delivered_orders_raw.csv")

In [2]:
df.head()

,order_id,price,order_estimated_delivery_date,order_delivered_customer_date,review_score,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,2017-09-29 00:00:00,2017-09-20 23:43:48,5.0,RJ
1,00018f77f2f0320c557190d7a144bdd3,239.90,2017-05-15 00:00:00,2017-05-12 16:04:24,4.0,SP
2,000229ec398224ef6ca0657da4fc703e,199.00,2018-02-05 00:00:00,2018-01-22 13:19:16,5.0,MG
3,00024acbcdf0a6daa1e931b038114c75,12.99,2018-08-20 00:00:00,2018-08-14 13:32:39,4.0,SP
4,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,2017-06-06 00:00:00,2017-05-22 13:44:35,4.0,MG


In [3]:
df.dtypes

order_id                             str
price                            float64
order_estimated_delivery_date        str
order_delivered_customer_date        str
review_score                     float64
customer_state                       str
dtype: object

In [4]:
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"])
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"])

In [5]:
df["delivery_delay_days"] = df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]

In [6]:
df["delivery_delay_days"].head()

0    -9 days +23:43:48
1    -3 days +16:04:24
2   -14 days +13:19:16
3    -6 days +13:32:39
4   -15 days +13:44:35
Name: delivery_delay_days, dtype: timedelta64[us]

In [7]:
df.dtypes

order_id                                     str
price                                    float64
order_estimated_delivery_date     datetime64[us]
order_delivered_customer_date     datetime64[us]
review_score                             float64
customer_state                               str
delivery_delay_days              timedelta64[us]
dtype: object

In [8]:
df["delivery_delay_days"] = df["delivery_delay_days"].dt.days

In [9]:
df["delivery_delay_days"].head()

0    -9.0
1    -3.0
2   -14.0
3    -6.0
4   -15.0
Name: delivery_delay_days, dtype: float64

In [10]:
df["delivery_delay_days"].isnull().sum()

np.int64(8)

In [11]:
df = df.dropna(subset=["delivery_delay_days"])

In [12]:
len(df)

110832

In [13]:
# Create a simple True/False flag: was this order late?
# This uses a "vectorized operation" — pandas checks every row's delivery_delay_days
# against the condition (> 0) all at once, and stores the True/False result per row.
df["is_late"] = df["delivery_delay_days"] > 0

In [14]:
# View delay number alongside its corresponding True/False flag, for the first 10 rows,
# to visually confirm the logic is working correctly
df[["delivery_delay_days", "is_late"]].head(10)

,delivery_delay_days,is_late
0,-9.0,False
1,-3.0,False
2,-14.0,False
3,-6.0,False
4,-15.0,False
5,0.0,False
6,-11.0,False
7,0.0,False
8,-22.0,False
9,-8.0,False


In [15]:
#.value_counts() counts how many times each unique value appears in a column — 
# perfect for a True/False column, since it'll tell us exactly
# how many orders were late vs. not late,
# across all 110,832 rows.
df["is_late"].value_counts()


is_late
False    103539
True       7293
Name: count, dtype: int64

In [16]:
# Save the cleaned DataFrame to a new CSV file, ready for Power BI to import
df.to_csv("../data/cleaned_orders.csv", index=False)

In [17]:
# Combine state code with "Brazil" for unambiguous map geocoding in Power BI
df["state_full"] = df["customer_state"] + ", Brazil"

# Re-save the cleaned file with this new column included
df.to_csv("../data/cleaned_orders.csv", index=False)

In [18]:
df["state_full"].head(10)

0    RJ, Brazil
1    SP, Brazil
2    MG, Brazil
3    SP, Brazil
4    MG, Brazil
5    SP, Brazil
6    SP, Brazil
7    SP, Brazil
8    RJ, Brazil
9    SP, Brazil
Name: state_full, dtype: str

In [19]:
df.columns

Index(['order_id', 'price', 'order_estimated_delivery_date',
       'order_delivered_customer_date', 'review_score', 'customer_state',
       'delivery_delay_days', 'is_late', 'state_full'],
      dtype='str')

In [20]:
# Dictionary mapping Brazilian state abbreviations to full state names,
# used so Power BI's map visual can correctly geocode locations
# (2-letter codes alone were being confused with other countries' abbreviations)
state_map = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapa",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceara",
    "DF": "Distrito Federal",
    "ES": "Espirito Santo",
    "GO": "Goias",
    "MA": "Maranhao",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Para",
    "PB": "Paraiba",
    "PR": "Parana",
    "PE": "Pernambuco",
    "PI": "Piaui",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondonia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "Sao Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins"
}

# Apply the mapping to convert each state code into its full name,
# then append ", Brazil" for unambiguous geocoding
df["state_full"] = df["customer_state"].map(state_map) + ", Brazil"

# Re-save the cleaned file with the corrected state_full column
df.to_csv("../data/cleaned_orders.csv", index=False)

In [21]:
df["state_full"].head()


0    Rio de Janeiro, Brazil
1         Sao Paulo, Brazil
2      Minas Gerais, Brazil
3         Sao Paulo, Brazil
4      Minas Gerais, Brazil
Name: state_full, dtype: str

In [22]:
df["state_full"].isnull().sum()

np.int64(0)

In [23]:
len(df)

110832

In [24]:
df["order_id"].duplicated().sum()

np.int64(14362)

In [25]:
df.groupby("order_id")["is_late"].nunique().gt(1).sum()

np.int64(0)

In [26]:
df[df["is_late"] == True]["order_id"].nunique()

6534